# MiniMax H3 — Combined Director + Refine + Voice on Colab

Optimized for **G4 / RTX PRO 6000 Blackwell 96 GB** and compatible with A100/L4/T4 fallback paths.

This setup matches the new combined workflow: **MiniMax H3 Director multi-clip timeline + Ref2V 8-step Turbo + optional reference voice/audio + H3 latent refine/upscale + automatic final video export**.


## 0. Mount Drive + persistence


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST_MODELS_TO_DRIVE = False   # False = faster loading from Colab VM disk
PERSIST_OUTPUT_TO_DRIVE = True    # Keep outputs / ComfyUI user state on Drive
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 1. Check GPU + choose safe runtime settings


In [ ]:
import subprocess, os, re
def sh(cmd): return subprocess.check_output(cmd, shell=True, text=True).strip()
gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
vram_gb = vram_mb / 1024
name = gpu_name.lower()
if 'rtx pro 6000' in name or 'blackwell' in name:
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    H3_RESERVE_VRAM_GB = '2'
else:
    H3_RESERVE_VRAM_GB = '1'
print(f'GPU: {gpu_name} ({vram_gb:.1f} GB)')
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 2. Clone our H3 branch


In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install / update ComfyUI + Director stack


In [ ]:
import os, subprocess, sys
os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['H3_PERSIST_MODELS'] = '1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT'] = '1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE'] = 'auto'
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!bash install_comfy_h3.sh

CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'):
        subprocess.run(['git','-C',path,'pull','--ff-only'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1',url,path], check=True)
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req): subprocess.run([sys.executable,'-m','pip','install','-r',req], check=False)

clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')
subprocess.run([sys.executable,'-m','pip','install','-U','huggingface_hub'], check=True)
print('Director / VHS / KJNodes / Pixaroma installed or updated.')


## 4. Download the correct H3 models

Quality path: **official Ref2VA INT8 ConvRot base + NVFP4 Qwen3-VL + Ref2V Turbo 8-step 768p + FP16 video VAE + FP32 audio VAE + H3 3D latent upscaler**.


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
MODEL_ROOT = Path(f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models']:
    (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)
downloads = [
 # repo, source filename, local_dir, final target
 ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),
 ('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', MODEL_ROOT, MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
 ('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),
 ('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),
 ('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),
 ('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_4step_v1.1_768p_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_4step_v1.1_768p_comfyui_bf16.safetensors'),
 ('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors', MODEL_ROOT/'latent_upscale_models', MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors'),
]
for repo_id, filename, local_dir, target in downloads:
    if target.exists() and target.stat().st_size > 1024*1024:
        print('SKIP', target.name)
        continue
    print('DOWNLOAD', filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir))
    if not target.exists():
        raise FileNotFoundError(f'Expected model was not created: {target}')
print('All combined-workflow models are ready at', MODEL_ROOT)


## 5. Install the finished Director + Refine workflow template

This copies the Director's current **second-pass / accelerated** example into your ComfyUI workflows folder, then patches it to the Ref2VA 8-step quality model stack used by our combined workflow. You can also upload the finished JSON from ChatGPT and overwrite it in the same folder.


In [ ]:
import json, pathlib
src = pathlib.Path('/content/ComfyUI/custom_nodes/ComfyUI_MiniMaxH3_Director/example_workflows/minimax_h3_director_二采_加速.json')
dst_dir = pathlib.Path('/content/ComfyUI/user/default/workflows')
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir/'Logan_H3_Combined_Director_Refine.json'
wf = json.loads(src.read_text(encoding='utf-8'))

# Main quality model: official Ref2VA INT8 ConvRot + Ref2V Turbo 8-step.
unets=[n for n in wf.get('nodes',[]) if n.get('type')=='UNETLoader']
for n in unets:
    vals=n.get('widgets_values',[])
    if vals: vals[0]='minimax_h3_ref2va_pruned_int8_convrot.safetensors'

loras=[n for n in wf.get('nodes',[]) if n.get('type')=='LoraLoaderModelOnly']
if loras:
    loras[0]['widgets_values']=['minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors',1.0]
    loras[0]['mode']=0
for n in loras[1:]:
    # Second pass uses the clean base model for quality rather than another Turbo pass.
    n['widgets_values']=['minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors',0.0]
    n['mode']=4

GLOBAL_PROMPT = '''subject_definitions:
<Subject 1> is Logan from <Picture 1>. Preserve his illustrated face, brown hair, proportions, forest-green hoodie with its white crown marking, necklace, black crossbody bag, and black-and-white cap with its green broken-bottle emblem and separated fragments. Preserve identity and illustration style across the episode.

If <Audio 1> is supplied globally, treat it as Logan's reference voice. Use it only when a segment asks for speech or narration. Prefer story-driven visible action, movement and interaction over static presenter shots.'''
fps=24; width=1344; height=768; seg_frames=192; seg_seconds=8
segments=[]
for i in range(4):
    segments.append({
      'id':f'logan_seg_{i+1}','start':i*seg_frames,'length':seg_frames,'frameCount':seg_frames,'durationSec':seg_seconds,
      'prompt':f'[CLIP {i+1} PLACEHOLDER] Use <Picture 1> for Logan identity. Add local Pictures 2-9 for this clip as needed (scene, prop, action/composition, details). Describe visible character movement, camera behavior and sound. Do not default to standing and explaining.',
      'negativePrompt':'','taskType':'','refs':[],'refAudios':[],'refVideos':[],
      'genImage':{'imageFile':'','fileName':''},'continuityFromPrev':False,'refImageSize':'match',
      'referenceVideo':{'videoFile':'','fileName':'','type':'input','subfolder':''},'_videoFrameCount':seg_frames,'previewFps':fps
    })
timeline={
 'version':5,'editMode':'segment','totalFrames':seg_frames*len(segments),'frameRate':fps,
 'video':{'fileName':'','videoFile':'','subfolder':'','type':'input','frames':[],'frameMap':[],'deletedSourceRanges':[],'sourceFrameCount':0,'width':0,'height':0},
 'videoClips':[],
 'global':{'taskType':'r2v — 参考主体生视频(Reference to Video)','prompt':GLOBAL_PROMPT,'refs':[],
           'referenceVideo':{'videoFile':'','fileName':'','type':'input','subfolder':''},'continuousReference':False,
           'genImage':{'imageFile':''},'sourceWidth':width,'sourceHeight':height,'refAudios':[],'commonEnabled':True,'commonCollapsed':False,'refVideos':[]},
 'output':{'mode':'fixed','aspectRatio':'16:9 (宽屏)','megapixels':0.98,'multiple':32,'longEdge':width,'width':width,'height':height,
           'maxExportFrames':0,'exportMode':'all','audioMode':'source','exportSourceImages':False,'refImageSize':'match',
           'continuityEnabled':True,'continuityOverlapFrames':22},
 'runSelectEnabled':True,'runSelection':[0],'segments':segments,
 'globalCommon':{'commonEnabled':True,'commonCollapsed':False,'prompt':GLOBAL_PROMPT,'refs':[],'refAudios':[],'refVideos':[]}
}
for n in wf.get('nodes',[]):
    if n.get('type')=='CLIPLoader': n['widgets_values']=['qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors','minimax','default']
    if n.get('type')=='VAELoader':
        title=n.get('title','').lower()
        if 'audio' in title: n['widgets_values']=['minimax_h3_audio_vae_fp32.safetensors']
        elif 'video' in title: n['widgets_values']=['minimax_h3_video_vae_fp16.safetensors']
    if n.get('type')=='MiniMaxH3Director':
        w=n.get('widgets_values',[])
        if len(w)>17:
            w[0]='r2v — 参考主体生视频(Reference to Video)'; w[1]=GLOBAL_PROMPT; w[3]=1; w[4]=0; w[5]='randomize'
            w[6]=fps; w[7]=width; w[8]=height; w[9]=width; w[10]=seg_frames*len(segments); w[11]=json.dumps(timeline,separators=(',',':'),ensure_ascii=False)
            w[13]=8; w[14]='euler'; w[15]='simple'; w[16]=6; w[17]=3
    if n.get('type')=='MiniMaxH3DirectorRefine':
        vals=n.get('widgets_values',[])
        for j,v in enumerate(vals):
            if isinstance(v,str) and ('latent_upscaler' in v.lower() or v.endswith('.safetensors') and '3d' in v.lower()):
                vals[j]='minimax_h3_latent_upscaler_3d_fp16.safetensors'
wf['id']='logan-h3-combined-director-refine'
dst.write_text(json.dumps(wf,ensure_ascii=False,indent=2),encoding='utf-8')
print('Workflow installed:',dst)
print('In Director: upload Global Picture 1 = Logan and Global Audio 1 = your voice reference.')


## 6. Launch ComfyUI


In [ ]:
!bash launch_comfy.sh


## First run

Open **`Logan_H3_Combined_Director_Refine.json`**.

Use **Global Picture 1** for Logan's master character image and **Global Audio 1** for the reference voice. Add scene / prop / motion refs locally per segment. Keep continuity disabled between unrelated scenes and enable it only for genuine continuations. Start at **1344×768, 24 FPS, 8 steps** on G4; if VRAM becomes tight, reduce the first-pass canvas to **1056×608 or 1024×576** while keeping the refine/upscale stage enabled.
